# Pre-processing & Re-Sampling

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import scipy.stats as stats
import itertools

def box_m_test(data, dvs, group_col):
    """Melakukan Box's M Test untuk menguji homogenitas matriks kovarians."""
    groups = data[group_col].unique()
    k = len(groups)
    p = len(dvs)
    if p == 0: return np.nan, np.nan, np.nan, np.nan
    N = len(data)
    n_i, cov_i, log_det_cov_i = [], [], []
    for group in groups:
        group_data = data[data[group_col] == group][dvs]
        if len(group_data) < p + 1 : return np.nan, np.nan, np.nan, np.nan
        ni = len(group_data)
        n_i.append(ni)
        cov = group_data.cov()
        cov_i.append(cov)
        sign, logdet = np.linalg.slogdet(cov)
        if sign <= 0: return np.nan, np.nan, np.nan, np.nan
        log_det_cov_i.append(logdet)
    sum_ni_minus_1_Si = sum((n - 1) * S for n, S in zip(n_i, cov_i))
    df_total = sum(n - 1 for n in n_i)
    if df_total <= 0: return np.nan, np.nan, np.nan, np.nan
    S_pooled = sum_ni_minus_1_Si / df_total
    sign_pooled, logdet_pooled = np.linalg.slogdet(S_pooled)
    if sign_pooled <= 0: return np.nan, np.nan, np.nan, np.nan
    sum_ni_minus_1_logdet = sum((n - 1) * logdet for n, logdet in zip(n_i, log_det_cov_i))
    M_statistic = df_total * logdet_pooled - sum_ni_minus_1_logdet
    sum_inv_ni_minus_1 = sum(1 / (n - 1) for n in n_i if n > 1)
    C = ((2 * p**2 + 3 * p - 1) / (6 * (p + 1) * (k - 1))) * (sum_inv_ni_minus_1 - 1/df_total)
    chi2_stat = M_statistic * (1 - C)
    df = (p * (p + 1) * (k - 1)) / 2
    p_val = stats.chi2.sf(chi2_stat, df)
    return M_statistic, chi2_stat, df, p_val

print("--- 1. Memuat Data ---")
try:
    df = pd.read_csv("C:/Uner/Semester 5/Multivariat/Coolyeah/Tugas Kelompok 2/SuperMarket Analysis.csv")
    df.columns = df.columns.str.strip() 
    df_processed = df.dropna().copy()
    print("Data Supermarket berhasil dimuat.\n")

    print("--- 2. Melakukan Pra-Pemrosesan pada Semua Kandidat DV ---")
    continuous_dvs_to_process = ['Unit price', 'Quantity', 'gross income', 'Rating']
    processed_dv_names = []

    for var in continuous_dvs_to_process:
        processed_var_name = f'{var.replace(" ", "_")}_processed'
        temp_series = df_processed[var]
        log_transformed = np.log1p(temp_series)
        Q1 = log_transformed.quantile(0.25)
        Q3 = log_transformed.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        capped = np.where(log_transformed < lower_bound, lower_bound, log_transformed)
        capped = np.where(capped > upper_bound, upper_bound, capped)
        df_processed[processed_var_name] = capped
        processed_dv_names.append(processed_var_name)
        print(f"Variabel '{var}' telah diproses -> '{processed_var_name}'")

    independent_candidates = ['City', 'Customer type', 'Gender', 'Payment']
    dependent_candidates = processed_dv_names

    print("\n--- 3. Memulai Pengujian Semua Kombinasi pada Data yang Telah Diproses ---")
    iv_combinations = list(itertools.combinations(independent_candidates, 2))
    dv_combinations = list(itertools.combinations(dependent_candidates, 3))
    results = []

    for iv_set in iv_combinations:
        for dv_set in dv_combinations:
            iv_set = list(iv_set)
            dv_set = list(dv_set)
            
            hz_result = pg.multivariate_normality(df_processed[dv_set], alpha=0.05)
            group_col_name = '_'.join(iv_set).replace(" ", "_")
            df_processed[group_col_name] = df_processed[iv_set].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)
            _, _, _, p_val_box = box_m_test(df_processed, dv_set, group_col_name)
            
            results.append({
                'Independent_Vars': ', '.join(iv_set),
                'Dependent_Vars': ', '.join(dv_set),
                'Multivariate_Normal': hz_result.normal,
                'HZ_Statistic': hz_result.hz,
                'Box_M_p_value': p_val_box
            })
            df_processed = df_processed.drop(columns=[group_col_name])

    print("\n--- 4. Hasil Pengujian Semua Kombinasi (Data Supermarket Setelah Pra-Pemrosesan) ---")
    results_df = pd.DataFrame(results).fillna(0)
    best_homogeneity = results_df.sort_values(by='Box_M_p_value', ascending=False)
    # Menampilkan 10 hasil teratas
    print(best_homogeneity.head(10))

    print("\n--- 5. REKOMENDASI ---")
    best_combo = best_homogeneity.iloc[0]
    print("Kombinasi yang paling mendekati asumsi (terutama homogenitas) adalah:")
    print(f"  -> Variabel Independen: {best_combo['Independent_Vars']}")
    print(f"  -> Variabel Dependen: {best_combo['Dependent_Vars']}")
    print(f"  -> P-value Box's M Test: {best_combo['Box_M_p_value']:.4f}")

except FileNotFoundError:
    print("ERROR: Pastikan file 'SuperMarket Analysis.csv' sudah diunggah.")
except Exception as e:
    print(f"Terjadi error: {e}")

--- 1. Memuat Data ---
Data Supermarket berhasil dimuat.

--- 2. Melakukan Pra-Pemrosesan pada Semua Kandidat DV ---
Variabel 'Unit price' telah diproses -> 'Unit_price_processed'
Variabel 'Quantity' telah diproses -> 'Quantity_processed'
Variabel 'gross income' telah diproses -> 'gross_income_processed'
Variabel 'Rating' telah diproses -> 'Rating_processed'

--- 3. Memulai Pengujian Semua Kombinasi pada Data yang Telah Diproses ---

--- 4. Hasil Pengujian Semua Kombinasi (Data Supermarket Setelah Pra-Pemrosesan) ---
          Independent_Vars                                     Dependent_Vars  \
2      City, Customer type  Unit_price_processed, gross_income_processed, ...   
3      City, Customer type  Quantity_processed, gross_income_processed, Ra...   
18  Customer type, Payment  Unit_price_processed, gross_income_processed, ...   
1      City, Customer type  Unit_price_processed, Quantity_processed, Rati...   
17  Customer type, Payment  Unit_price_processed, Quantity_processed, Ra

# BOX M Test & membuat File Baru

In [ ]:
def box_m_test(data, dvs, group_col):
    """Melakukan Box's M Test untuk menguji homogenitas matriks kovarians."""
    groups = data[group_col].unique()
    k = len(groups)
    p = len(dvs)
    if p == 0: return np.nan, np.nan, np.nan, np.nan
    N = len(data)
    n_i, cov_i, log_det_cov_i = [], [], []
    for group in groups:
        group_data = data[data[group_col] == group][dvs]
        if len(group_data) < p + 1 : return np.nan, np.nan, np.nan, np.nan
        ni = len(group_data)
        n_i.append(ni)
        cov = group_data.cov()
        cov_i.append(cov)
        sign, logdet = np.linalg.slogdet(cov)
        if sign <= 0: return np.nan, np.nan, np.nan, np.nan
        log_det_cov_i.append(logdet)
    sum_ni_minus_1_Si = sum((n - 1) * S for n, S in zip(n_i, cov_i))
    df_total = sum(n - 1 for n in n_i)
    if df_total <= 0: return np.nan, np.nan, np.nan, np.nan
    S_pooled = sum_ni_minus_1_Si / df_total
    sign_pooled, logdet_pooled = np.linalg.slogdet(S_pooled)
    if sign_pooled <= 0: return np.nan, np.nan, np.nan, np.nan
    sum_ni_minus_1_logdet = sum((n - 1) * logdet for n, logdet in zip(n_i, log_det_cov_i))
    M_statistic = df_total * logdet_pooled - sum_ni_minus_1_logdet
    sum_inv_ni_minus_1 = sum(1 / (n - 1) for n in n_i if n > 1)
    C = ((2 * p**2 + 3 * p - 1) / (6 * (p + 1) * (k - 1))) * (sum_inv_ni_minus_1 - 1/df_total)
    chi2_stat = M_statistic * (1 - C)
    df = (p * (p + 1) * (k - 1)) / 2
    p_val = stats.chi2.sf(chi2_stat, df)
    return M_statistic, chi2_stat, df, p_val

print("--- 1. Memuat Data ---")
try:
    df = pd.read_csv("C:/Uner/Semester 5/Multivariat/Coolyeah/Tugas Kelompok 2/SuperMarket Analysis.csv")
    df.columns = df.columns.str.strip()
    df_processed = df.dropna().copy()
    print("Data Supermarket berhasil dimuat.\n")

    print("--- 2. Melakukan Pra-Pemrosesan pada Semua Kandidat DV ---")
    continuous_dvs_to_process = ['Unit price', 'Quantity', 'gross income', 'Rating']
    processed_dv_names = []
    for var in continuous_dvs_to_process:
        processed_var_name = f'{var.replace(" ", "_")}_processed'
        temp_series = df_processed[var]
        log_transformed = np.log1p(temp_series)
        Q1 = log_transformed.quantile(0.25)
        Q3 = log_transformed.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        capped = np.where(log_transformed < lower_bound, lower_bound, log_transformed)
        capped = np.where(capped > upper_bound, upper_bound, capped)
        df_processed[processed_var_name] = capped
        processed_dv_names.append(processed_var_name)
    print("Pra-pemrosesan selesai.\n")

    print("--- 3. Memulai Pengujian Semua Kombinasi ---")
    independent_candidates = ['City', 'Customer type', 'Gender', 'Payment']
    dependent_candidates = processed_dv_names
    iv_combinations = list(itertools.combinations(independent_candidates, 2))
    dv_combinations = list(itertools.combinations(dependent_candidates, 3))
    results = []
    for iv_set in iv_combinations:
        for dv_set in dv_combinations:
            iv_set = list(iv_set)
            dv_set = list(dv_set)
            hz_result = pg.multivariate_normality(df_processed[dv_set], alpha=0.05)
            group_col_name = '_'.join(iv_set).replace(" ", "_")
            df_processed[group_col_name] = df_processed[iv_set].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)
            _, _, _, p_val_box = box_m_test(df_processed, dv_set, group_col_name)
            results.append({
                'Independent_Vars': ', '.join(iv_set),
                'Dependent_Vars': ', '.join(dv_set),
                'Multivariate_Normal': hz_result.normal,
                'HZ_Statistic': hz_result.hz,
                'Box_M_p_value': p_val_box
            })
            df_processed = df_processed.drop(columns=[group_col_name])

    print("--- 4. Menganalisis Hasil Kombinasi ---")
    results_df = pd.DataFrame(results).fillna(0)
    best_homogeneity = results_df.sort_values(by='Box_M_p_value', ascending=False)
    best_combo = best_homogeneity.iloc[0]
    print("Kombinasi terbaik ditemukan.\n")

    print("--- 5. Menyimpan DataFrame Baru Sesuai Kombinasi Terbaik ---")
    
    best_ivs_str = best_combo['Independent_Vars']
    best_dvs_str = best_combo['Dependent_Vars']
    
    best_ivs_list = [iv.strip() for iv in best_ivs_str.split(',')]
    best_dvs_list = [dv.strip() for dv in best_dvs_str.split(',')]

    final_columns_to_save = best_ivs_list + best_dvs_list

    df_final = df_processed[final_columns_to_save].copy()
    
    output_filename = 'SuperMarket_Data_Terbaik_Untuk_MANOVA.csv'
    df_final.to_csv(output_filename, index=False)
    
    print(f"DataFrame baru berhasil disimpan sebagai: '{output_filename}'")
    print("\nFile ini berisi kolom:")
    for col in df_final.columns:
        print(f"- {col}")

except FileNotFoundError:
    print("ERROR: Pastikan file 'SuperMarket Analysis.csv' sudah diunggah.")
except Exception as e:
    print(f"Terjadi error: {e}")

--- 1. Memuat Data ---
Data Supermarket berhasil dimuat.

--- 2. Melakukan Pra-Pemrosesan pada Semua Kandidat DV ---
Pra-pemrosesan selesai.

--- 3. Memulai Pengujian Semua Kombinasi ---
--- 4. Menganalisis Hasil Kombinasi ---
Kombinasi terbaik ditemukan.

--- 5. Menyimpan DataFrame Baru Sesuai Kombinasi Terbaik ---
DataFrame baru berhasil disimpan sebagai: 'SuperMarket_Data_Terbaik_Untuk_MANOVA.csv'

File ini berisi kolom:
- City
- Customer type
- Unit_price_processed
- gross_income_processed
- Rating_processed
